# Histoformer — a from-the-equations PyTorch re-implementation

**Paper:** *Restoring Images in Adverse Weather Conditions via Histogram Transformer* — Shangquan Sun, Wenqi Ren, Xinwei Gao, Rui Wang, Xiaochun Cao. ECCV 2024 (arXiv:2407.10172).
**Official code:** <https://github.com/sunshangquan/Histoformer>

This notebook rebuilds Histoformer from the paper's equations (Eq. 1–10) and Figure 2, and uses the official repository to confirm details the paper leaves open. It is self-contained: everything below runs from this file alone (the official repo is only imported in the optional equivalence check, Section 11).

**Layout**

| § | Section | Paper reference |
|---|---|---|
| 0 | Imports & config (all hyper-parameters) | §4.1 *Training details*, Fig. 2 |
| 1 | Backbone building blocks (LayerNorm, patch embed, crude skip, down/up-sample) | §3.1, Fig. 2 (left) |
| 2 | Dynamic-range convolution | §3.2, Eq. 3, Fig. 2 (DHSA, left part) |
| 3 | Histogram self-attention (BHR + FHR) → DHSA | §3.2, Eq. 4–5, Fig. 2 (DHSA) |
| 4 | Dual-scale Gated Feed-Forward (DGFF) | §3.2, Eq. 6, Fig. 2 (DGFF) |
| 5 | Histogram Transformer Block (HTB) | §3.2, Eq. 1–2, Fig. 2 (HTB) |
| 6 | Full Histoformer U-shaped model | §3.1, Fig. 2 |
| 7 | Matcher / assignment | — (not used by this method) |
| 8 | Losses: L1 reconstruction, Pearson correlation, total | §3.3, Eq. 7–10 |
| 9 | Dataset + augmentation (+ progressive patch schedule) | §4.1 *Datasets*, *Training details* |
| 10 | Inference & evaluation metrics (PSNR / SSIM on Y) | §4.2, Table 1 protocol |
| 11 | Training loop (defined, **not run**) | §4.1 *Training details* |
| 12 | Sanity checks + equivalence with the official code | — |

### Where the official repo differs from the paper

The default configuration (`PAPER` preset) follows the paper. Every point where the repository does something different is exposed as a switch, and the `REPO` preset turns all of them on so the official checkpoint loads (`strict=True`) and the forward pass matches the official `Histoformer` exactly (verified in §12).

| # | Item | Paper (default here) | Official repo | Switch |
|---|---|---|---|---|
| 1 | Attention logits in Eq. 5 | `softmax(QKᵀ/√k)`, k = number of heads | L2-normalise Q and K along the feature axis, multiply by a learnable per-head `temperature`, then a "softmax + 1" (`exp(x)/(Σexp(x)+1)`) | `attn_mode = 'paper' / 'repo'` |
| 2 | Where pixels are "sorted back" after dynamic-range conv | **Before** the final 1×1 projection (text of §3.2 and Fig. 2: Scatter → Concat → 1×1) | **After** `project_out` | `unsort_before_proj = True / False` |
| 3 | Decoder level 1 | Fig. 2 shows a 1×1 conv after the level-1 concat that brings the width back to C (the level-1 decoder, refinement and output conv then run at C) | No 1×1 reduction: level-1 decoder, 4 refinement blocks and output conv run at 2C (Restormer layout) | `dec1_reduce = True / False` |
| 4 | Data augmentation | Random horizontal and vertical flips (§4.1) | 8-way dihedral augmentation (flips + 90° rotations, `random_augmentation`) | `aug_mode = 'flip' / 'dihedral'` |
| 5 | Pearson loss over a batch | Eq. 8–9 per image, averaged | Same, but samples whose loss is NaN/Inf are dropped and `1e-6` is added to the denominator | `pearson_repo_style = False / True` |

Details the paper does **not** specify, taken from the repo and used in both presets: the number of bins in DHSA equals the number of heads of that stage (`factor = num_heads`); sequences are zero-padded to a multiple of the bin count; LayerNorm has no affine parameters; 4 refinement HTBs after the decoder; dilation 2 for the dilated 3×3 in DGFF; AdamW weight decay 1e-4, β = (0.9, 0.999); gradient-norm clipping at 0.01; progressive patch/batch schedule; test-time reflect padding to a multiple of 8; PSNR/SSIM on the Y channel. The sort order in dynamic-range convolution follows Fig. 2 and the repo (sort along H, then along W); the reverse order is available as `sort_order='wh'` (Table 2 of the paper compares both orders).

## 0 · Imports and configuration
*Paper §4.1 "Training details"; architecture numbers from §4.1 and Fig. 2.*

All numbers stated in the paper are marked `# paper`. Numbers the paper leaves open are marked `# repo` (taken from `Allweather/Options/Allweather_Histoformer.yml` and the training code).

In [1]:
import math, os, random, glob, copy, time, numbers
from dataclasses import dataclass, field, asdict, replace
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', DEVICE)

torch 2.15.0+cu132 | device: cpu


In [2]:
@dataclass
class HistoformerConfig:
    # ---------------- architecture ----------------
    inp_channels: int = 3
    out_channels: int = 3
    dim: int = 36                                  # paper: channel size C = 36
    num_blocks: List[int] = field(default_factory=lambda: [4, 4, 6, 8])   # paper: L1..L4
    heads: List[int] = field(default_factory=lambda: [1, 2, 4, 8])        # paper: heads per stage
    ffn_expansion_factor: float = 2.667            # paper: DGFF expansion r
    num_refinement_blocks: int = 4                 # repo
    bias: bool = False                             # repo
    LayerNorm_type: str = 'WithBias'               # repo (no affine params)
    # ---------------- paper-vs-repo switches ----------------
    attn_mode: str = 'paper'            # 'paper' : softmax(QK^T/sqrt(k)) | 'repo': L2-norm + temperature + softmax_1
    unsort_before_proj: bool = True     # paper: scatter back before the final 1x1 | repo: after it
    dec1_reduce: bool = True            # paper (Fig. 2): 1x1 to C after level-1 concat | repo: stay at 2C
    sort_order: str = 'hw'              # 'hw': sort along H then W (Fig. 2 & repo) | 'wh': reverse
    # ---------------- losses ----------------
    alpha_cor: float = 1.0              # paper: weight of correlation loss (Table 6 -> 1)
    pearson_repo_style: bool = False    # repo drops NaN/Inf samples and adds 1e-6
    # ---------------- data ----------------
    aug_mode: str = 'flip'              # paper: random H/V flips | repo: 'dihedral'
    # ---------------- optimisation ----------------
    total_iters: int = 300_000          # paper
    lr: float = 3e-4                    # paper
    lr_min: float = 1e-6                # paper
    const_iters: int = 92_000           # paper: constant LR for the first 92k
    cosine_iters: int = 208_000         # paper: cosine to 1e-6 over the remaining 208k
    weight_decay: float = 1e-4          # repo
    betas: tuple = (0.9, 0.999)         # repo
    grad_clip: float = 0.01             # repo (clip_grad_norm_ max norm)
    batch_size: int = 8                 # paper: initial batch size
    patch_size: int = 128               # paper: initial patch size
    # progressive learning (paper: "akin to the progressive learning pipeline"; values from repo)
    prog_iters: List[int] = field(default_factory=lambda: [92000, 84000, 56000, 36000, 32000])
    prog_batch: List[int] = field(default_factory=lambda: [8, 5, 2, 1, 1])
    prog_patch: List[int] = field(default_factory=lambda: [128, 160, 256, 320, 362])
    # ---------------- evaluation ----------------
    test_pad_factor: int = 8            # repo: reflect-pad to a multiple of 8
    eval_y_channel: bool = True         # repo: PSNR/SSIM on Y of YCbCr
    ssim_win: int = 11                  # repo

PAPER = HistoformerConfig()
REPO  = replace(PAPER, attn_mode='repo', unsort_before_proj=False, dec1_reduce=False,
                aug_mode='dihedral', pearson_repo_style=True)
CFG = PAPER
for k, v in asdict(CFG).items():
    if getattr(REPO, k) != v: print(f'{k:22s} paper={v!s:10s} repo={getattr(REPO, k)}')

attn_mode              paper=paper      repo=repo
unsort_before_proj     paper=True       repo=False
dec1_reduce            paper=True       repo=False
pearson_repo_style     paper=False      repo=True
aug_mode               paper=flip       repo=dihedral


## 1 · Backbone building blocks
*Paper §3.1 "Overall Architecture", Fig. 2 (left half).*

* **Overlapping patch embedding:** a 3×3 conv maps the image $I^{lq}\in\mathbb R^{3\times H\times W}$ to $C$ channels.
* **Down/up-sampling:** pixel-unshuffle / pixel-shuffle between stages (a 3×3 conv first halves/doubles channels so each step gives 2× / ½× width).
* **Crude skip-connection:** for every encoder stage after the first, the input image is passed through *AvgPool → 1×1 conv (pixel-wise) → 3×3 depth-wise conv*, concatenated with the down-sampled features, then fused with a 1×1 conv. The pooling is chained, so stage *s* sees the image at $1/2^{s}$ resolution.
* **LayerNorm** over channels at every pixel (Eq. 1–2). As in the repo it has no affine parameters.

In [3]:
def to_3d(x):       return rearrange(x, 'b c h w -> b (h w) c')
def to_4d(x, h, w): return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5)

class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5)

class LayerNorm(nn.Module):
    '''Channel-wise LayerNorm on a (B,C,H,W) map — "LN" in Eq. 1-2.'''
    def __init__(self, dim, LayerNorm_type='WithBias'):
        super().__init__()
        self.body = BiasFree_LayerNorm(dim) if LayerNorm_type == 'BiasFree' else WithBias_LayerNorm(dim)
    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)

class OverlapPatchEmbed(nn.Module):
    '''3x3 conv patch embedding (§3.1).'''
    def __init__(self, in_c=3, embed_dim=48, bias=False):
        super().__init__()
        self.proj = nn.Conv2d(in_c, embed_dim, kernel_size=3, stride=1, padding=1, bias=bias)
    def forward(self, x): return self.proj(x)

class SkipPatchEmbed(nn.Module):
    '''Crude skip-connection: AvgPool -> 1x1 conv -> 3x3 depth-wise conv (§3.1).'''
    def __init__(self, in_c=3, dim=48, bias=False):
        super().__init__()
        self.proj = nn.Sequential(
            nn.AvgPool2d(2, stride=2, padding=0, ceil_mode=False, count_include_pad=True),
            nn.Conv2d(in_c, dim, kernel_size=1, bias=bias),
            nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=bias))
    def forward(self, x): return self.proj(x)

class Downsample(nn.Module):
    '''conv3x3 (C -> C/2) + PixelUnshuffle(2): (C,H,W) -> (2C,H/2,W/2).'''
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(n_feat, n_feat // 2, 3, 1, 1, bias=False), nn.PixelUnshuffle(2))
    def forward(self, x): return self.body(x)

class Upsample(nn.Module):
    '''conv3x3 (C -> 2C) + PixelShuffle(2): (C,H,W) -> (C/2,2H,2W).'''
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(n_feat, n_feat * 2, 3, 1, 1, bias=False), nn.PixelShuffle(2))
    def forward(self, x): return self.body(x)

x = torch.randn(2, 36, 32, 32)
print('LayerNorm   ', tuple(LayerNorm(36)(x).shape))
print('Downsample  ', tuple(Downsample(36)(x).shape))
print('Upsample    ', tuple(Upsample(36)(x).shape))
print('SkipPatchEmbed on image', tuple(SkipPatchEmbed(3, 3)(torch.randn(2, 3, 32, 32)).shape))

LayerNorm    (2, 36, 32, 32)
Downsample   (2, 72, 16, 16)
Upsample     (2, 18, 64, 64)
SkipPatchEmbed on image (2, 3, 16, 16)


## 2 · Dynamic-range convolution
*Paper §3.2 "Dynamic-range Convolution", Eq. 3, Fig. 2 (DHSA, "Sort along H / Sort along W" → 1×1 → 3×3 DWConv).*

$$
F_1, F_2 = \mathrm{Split}(F),\qquad F_1 = \mathrm{Sort_v}(\mathrm{Sort_h}(F_1)),\qquad
F = \mathrm{Conv}^{d}_{3\times3}\big(\mathrm{Conv}_{1\times1}(\mathrm{Concat}(F_1,F_2))\big)
$$

Half of the channels are sorted along one spatial axis and then the other, so bright and dark values gather in opposite corners. A normal 3×3 kernel applied afterwards then mixes pixels that are similar in intensity rather than pixels that are merely neighbours. The other half is left untouched so local structure is still available.

The 1×1 + 3×3 depth-wise pair produces all five tensors needed by the attention (two query–key pairs and the value, Eq. 4), so it outputs $5C$ channels. The sorting indices are returned so the features can be "sorted back" later.

In [4]:
def sort_spatial(x_half, order='hw'):
    '''Sort a (B,C',H,W) tensor along H then W ('hw') or W then H ('wh'). Returns sorted tensor + indices.'''
    if order == 'hw':
        xs, idx_a = x_half.sort(-2)        # along H (each column)
        xs, idx_b = xs.sort(-1)            # along W (each row)
    else:
        xs, idx_a = x_half.sort(-1)
        xs, idx_b = xs.sort(-2)
    return xs, (idx_a, idx_b)

def unsort_spatial(x_half, idxs, order='hw'):
    '''Inverse of sort_spatial: scatter values back to their original positions (last sort undone first).'''
    idx_a, idx_b = idxs
    dims = (-2, -1) if order == 'hw' else (-1, -2)
    x_half = torch.scatter(x_half, dims[1], idx_b, x_half)
    x_half = torch.scatter(x_half, dims[0], idx_a, x_half)
    return x_half

# quick property check: unsort(sort(x)) == x and the sorted map is monotone in both directions
t = torch.randn(1, 2, 5, 6)
ts, idx = sort_spatial(t)
assert torch.equal(unsort_spatial(ts.clone(), idx), t)
print('rows sorted:', bool((ts.diff(dim=-1) >= 0).all()), '| columns sorted:', bool((ts.diff(dim=-2) >= 0).all()))
print(ts[0, 0].numpy().round(2))

rows sorted: True | columns sorted: True
[[-1.99 -1.78 -1.31 -1.03 -0.98 -0.73]
 [-1.35 -0.55 -0.52 -0.5  -0.4   0.27]
 [-0.4  -0.13  0.05  0.46  0.54  0.76]
 [-0.22  0.8   1.    1.04  1.35  1.51]
 [ 0.28  1.26  1.34  1.4   1.78  1.82]]


## 3 · Histogram self-attention and the DHSA module
*Paper §3.2 "Histogram Self-Attention", Eq. 4–5, Fig. 2 (DHSA, BHR and FHR panels).*

**Sorting into a histogram (Eq. 4).** The value map $V$ is flattened to $C\times HW$ and sorted per channel; the same permutation $d$ is gathered onto both query–key pairs:

$$
V, d = \mathrm{Sort}\big(\mathbf R^{C\times HW}_{C\times H\times W}(V)\big),\quad
Q_i, K_i = \mathrm{Split}\big(\mathrm{Gather}(\mathbf R(F_{QK,i}), d)\big),\ i\in\{1,2\}
$$

**Two reshapes (Eq. 5).** With $B$ bins the sorted sequence becomes $C\times B\times HW/B$:

* **BHR (bin-wise):** $B$ bins of $HW/B$ consecutive (in intensity) pixels each → coarse, global grouping. In code: `'(factor hw)'`.
* **FHR (frequency-wise):** $HW/B$ bins of $B$ pixels each → fine-grained grouping of near-identical intensities. In code: `'(hw factor)'`.

$$
A_B = \mathrm{softmax}\!\Big(\tfrac{\mathbf R_B(Q_1)\mathbf R_B(K_1)^\top}{\sqrt k}\Big)\mathbf R_B(V),\qquad
A_F = \mathrm{softmax}\!\Big(\tfrac{\mathbf R_F(Q_2)\mathbf R_F(K_2)^\top}{\sqrt k}\Big)\mathbf R_F(V),\qquad
A = A_B\odot A_F
$$

Tokens are the (channel, bin) pairs within a head and their features run along the pixels in the bin, so the attention matrix is $(C_h B)\times(C_h B)$ — linear in $HW$, which is why this works at full resolution. After attention, the outputs are scattered back to their original pixel positions, the dynamic-range sort is undone, and a 1×1 conv projects the result.

`attn_mode='paper'` uses the plain scaled softmax of Eq. 5. `attn_mode='repo'` reproduces the official code (L2-normalised Q/K, learnable temperature, `softmax_1`). The `temperature` parameter always exists so checkpoints load in either mode.

In [5]:
class Attention_histogram(nn.Module):
    '''Dynamic-range Histogram Self-Attention (DHSA), Eq. 3-5.'''
    def __init__(self, dim, num_heads, bias, ifBox=True, attn_mode='paper',
                 unsort_before_proj=True, sort_order='hw'):
        super().__init__()
        self.factor = num_heads            # number of bins B (repo: B = num_heads of the stage)
        self.ifBox = ifBox
        self.num_heads = num_heads
        self.attn_mode, self.unsort_before_proj, self.sort_order = attn_mode, unsort_before_proj, sort_order
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))   # used in 'repo' mode
        # Eq. 3: Conv1x1 then 3x3 depth-wise conv, producing Q1,K1,Q2,K2,V (5C channels)
        self.qkv = nn.Conv2d(dim, dim * 5, kernel_size=1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim * 5, dim * 5, kernel_size=3, stride=1, padding=1, groups=dim * 5, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    # ---- helpers -------------------------------------------------------
    def pad(self, x, factor):
        hw = x.shape[-1]
        t_pad = [0, 0] if hw % factor == 0 else [0, (hw // factor + 1) * factor - hw]
        return F.pad(x, t_pad, 'constant', 0), t_pad

    def unpad(self, x, t_pad):
        return x[:, :, t_pad[0]:x.shape[-1] - t_pad[1]]

    @staticmethod
    def softmax_1(x, dim=-1):            # repo: exp(x) / (sum exp(x) + 1)
        logit = x.exp()
        return logit / (logit.sum(dim, keepdim=True) + 1)

    def reshape_attn(self, q, k, v, ifBox):
        '''Histogram reshaping (BHR if ifBox else FHR) + self-attention, Eq. 5.'''
        b, c = q.shape[:2]
        q, t_pad = self.pad(q, self.factor)
        k, _ = self.pad(k, self.factor)
        v, _ = self.pad(v, self.factor)
        hw = q.shape[-1] // self.factor
        shape_ori = 'b (head c) (factor hw)' if ifBox else 'b (head c) (hw factor)'   # BHR | FHR
        shape_tar = 'b head (c factor) hw'
        q, k, v = (rearrange(t, f'{shape_ori} -> {shape_tar}', factor=self.factor, hw=hw, head=self.num_heads)
                   for t in (q, k, v))
        if self.attn_mode == 'repo':
            q = F.normalize(q, dim=-1); k = F.normalize(k, dim=-1)
            attn = self.softmax_1((q @ k.transpose(-2, -1)) * self.temperature, dim=-1)
        else:                             # paper Eq. 5: softmax(Q K^T / sqrt(k)), k = number of heads
            attn = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.num_heads), dim=-1)
        out = attn @ v
        out = rearrange(out, f'{shape_tar} -> {shape_ori}', factor=self.factor, hw=hw, b=b, head=self.num_heads)
        return self.unpad(out, t_pad)

    # ---- forward -------------------------------------------------------
    def forward(self, x):
        b, c, h, w = x.shape
        # Eq. 3: dynamic-range convolution
        x1_sorted, idx_sp = sort_spatial(x[:, :c // 2], self.sort_order)
        x = torch.cat([x1_sorted, x[:, c // 2:]], dim=1)
        qkv = self.qkv_dwconv(self.qkv(x))
        q1, k1, q2, k2, v = qkv.chunk(5, dim=1)
        # Eq. 4: sort V per channel, gather Q/K with the same index
        v, idx = v.reshape(b, c, -1).sort(dim=-1)
        q1, k1, q2, k2 = (torch.gather(t.reshape(b, c, -1), 2, idx) for t in (q1, k1, q2, k2))
        # Eq. 5: BHR and FHR branches, scatter back to pixel order, element-wise product
        out1 = self.reshape_attn(q1, k1, v, True)
        out2 = self.reshape_attn(q2, k2, v, False)
        out1 = torch.scatter(out1, 2, idx, out1).view(b, c, h, w)
        out2 = torch.scatter(out2, 2, idx, out2).view(b, c, h, w)
        out = out1 * out2
        if self.unsort_before_proj:      # paper: Scatter -> Concat -> 1x1 (Fig. 2)
            out = torch.cat([unsort_spatial(out[:, :c // 2], idx_sp, self.sort_order), out[:, c // 2:]], 1)
            return self.project_out(out)
        out = self.project_out(out)      # repo: 1x1 first, then un-sort the first half
        return torch.cat([unsort_spatial(out[:, :c // 2], idx_sp, self.sort_order), out[:, c // 2:]], 1)

for mode in ('paper', 'repo'):
    att = Attention_histogram(72, num_heads=2, bias=False, attn_mode=mode, unsort_before_proj=(mode == 'paper'))
    y = att(torch.randn(2, 72, 16, 20))   # HW=320, divisible by B=2
    print(f'DHSA[{mode}]  in (2,72,16,20) -> out {tuple(y.shape)}  finite={bool(torch.isfinite(y).all())}')
y = Attention_histogram(144, num_heads=4, bias=False)(torch.randn(1, 144, 7, 9))   # HW=63 -> zero-padded to 64
print('DHSA with padding (HW=63, B=4):', tuple(y.shape))

DHSA[paper]  in (2,72,16,20) -> out (2, 72, 16, 20)  finite=True
DHSA[repo]  in (2,72,16,20) -> out (2, 72, 16, 20)  finite=True
DHSA with padding (HW=63, B=4): (1, 144, 7, 9)


## 4 · Dual-scale Gated Feed-Forward (DGFF)
*Paper §3.2 "Dual-scale Gated Feed-Forward", Eq. 6, Fig. 2 (DGFF panel).*

$$
F_{l,1},F_{l,2}=\mathrm{Split}\big(\mathrm{Shuffle}(\mathrm{Conv}_{1\times1}(F_l))\big),\quad
F_{l,1}=\mathrm{Conv}^{d}_{5\times5}(F_{l,1}),\quad F_{l,2}=\mathrm{Conv}^{d,\text{dilated}}_{3\times3}(F_{l,2}),
$$
$$
F_{l+1}=\mathrm{Conv}_{1\times1}\big(\mathrm{Unshuffle}(\mathrm{Mish}(F_{l,2})\odot F_{l,1})\big)
$$

A 1×1 conv expands to $2rC$ channels; pixel-shuffle trades channels for a 2× larger grid (so the depth-wise kernels see a finer scale). One branch uses a 5×5 depth-wise conv, the other a dilated 3×3 (dilation 2, receptive field 5×5 with a sparser footprint). The Mish-activated dilated branch gates the 5×5 branch, then pixel-unshuffle and a 1×1 conv bring it back to $C$.

In [6]:
class FeedForward(nn.Module):
    '''Dual-scale Gated Feed-Forward (DGFF), Eq. 6.'''
    def __init__(self, dim, ffn_expansion_factor, bias):
        super().__init__()
        hidden_features = int(dim * ffn_expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden_features * 2, kernel_size=1, bias=bias)
        self.dwconv_5 = nn.Conv2d(hidden_features // 4, hidden_features // 4, kernel_size=5, stride=1, padding=2,
                                  groups=hidden_features // 4, bias=bias)
        self.dwconv_dilated2_1 = nn.Conv2d(hidden_features // 4, hidden_features // 4, kernel_size=3, stride=1,
                                           padding=2, groups=hidden_features // 4, bias=bias, dilation=2)
        self.p_unshuffle = nn.PixelUnshuffle(2)
        self.p_shuffle = nn.PixelShuffle(2)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.p_shuffle(self.project_in(x))          # Conv1x1 -> Shuffle
        x1, x2 = x.chunk(2, dim=1)                       # Split
        x1 = self.dwconv_5(x1)                           # 5x5 depth-wise
        x2 = self.dwconv_dilated2_1(x2)                  # dilated 3x3 depth-wise
        x = F.mish(x2) * x1                              # gating
        return self.project_out(self.p_unshuffle(x))     # Unshuffle -> Conv1x1

ff = FeedForward(36, 2.667, False)
print('DGFF hidden =', int(36 * 2.667), '| out', tuple(ff(torch.randn(2, 36, 16, 16)).shape))

DGFF hidden = 96 | out (2, 36, 16, 16)


## 5 · Histogram Transformer Block (HTB)
*Paper §3.2, Eq. 1–2, Fig. 2 (HTB panel).*

$$
F_l = F_{l-1} + \mathrm{DHSA}(\mathrm{LN}(F_{l-1})),\qquad F_l = F_l + \mathrm{DGFF}(\mathrm{LN}(F_l))
$$

In [7]:
class TransformerBlock(nn.Module):
    '''Histogram Transformer Block, Eq. 1-2. Attribute names match the official repo.'''
    def __init__(self, dim, num_heads, ffn_expansion_factor, bias, LayerNorm_type, cfg=None):
        super().__init__()
        cfg = cfg or CFG
        self.attn_g = Attention_histogram(dim, num_heads, bias, True, attn_mode=cfg.attn_mode,
                                          unsort_before_proj=cfg.unsort_before_proj, sort_order=cfg.sort_order)
        self.norm_g = LayerNorm(dim, LayerNorm_type)
        self.ffn = FeedForward(dim, ffn_expansion_factor, bias)
        self.norm_ff1 = LayerNorm(dim, LayerNorm_type)

    def forward(self, x):
        x = x + self.attn_g(self.norm_g(x))       # Eq. 1
        return x + self.ffn(self.norm_ff1(x))     # Eq. 2

htb = TransformerBlock(36, 1, 2.667, False, 'WithBias')
print('HTB out', tuple(htb(torch.randn(2, 36, 16, 16)).shape), '| params', sum(p.numel() for p in htb.parameters()))

HTB out (2, 36, 16, 16) | params 20581


## 6 · Full Histoformer
*Paper §3.1, Fig. 2 (left two columns).*

A four-level encoder–decoder with $L=\{4,4,6,8\}$ HTBs, widths $C,2C,4C,8C$ ($C=36$) and $\{1,2,4,8\}$ heads. Encoder stages 2–4 receive the crude skip-connection from the image; decoder stages concatenate the matching encoder features and fuse them with a 1×1 conv. Four refinement HTBs follow the decoder, a 3×3 conv maps back to RGB, and the network predicts a residual that is added to the input:

$$ I^{hq} = I^{lq} + \mathrm{Conv}_{3\times3}(\text{decoder}(\cdot)) $$

With `dec1_reduce=True` (Fig. 2) the level-1 concatenation is reduced to $C$ by a 1×1 conv named `reduce_chan_level1`; with `False` (repo) the level-1 decoder, refinement and output layers work at $2C$.

In [8]:
class Histoformer(nn.Module):
    def __init__(self, inp_channels=3, out_channels=3, dim=36, num_blocks=(4, 4, 6, 8), num_refinement_blocks=4,
                 heads=(1, 2, 4, 8), ffn_expansion_factor=2.667, bias=False, LayerNorm_type='WithBias',
                 dual_pixel_task=False, cfg: Optional[HistoformerConfig] = None):
        super().__init__()
        cfg = cfg or CFG
        self.cfg = cfg
        blk = lambda d, h, n: nn.Sequential(*[TransformerBlock(d, h, ffn_expansion_factor, bias, LayerNorm_type, cfg)
                                             for _ in range(n)])
        self.patch_embed = OverlapPatchEmbed(inp_channels, dim)
        # ---- encoder
        self.encoder_level1 = blk(dim, heads[0], num_blocks[0])
        self.down1_2 = Downsample(dim)
        self.encoder_level2 = blk(dim * 2, heads[1], num_blocks[1])
        self.down2_3 = Downsample(dim * 2)
        self.encoder_level3 = blk(dim * 4, heads[2], num_blocks[2])
        self.down3_4 = Downsample(dim * 4)
        self.latent = blk(dim * 8, heads[3], num_blocks[3])
        # ---- decoder
        self.up4_3 = Upsample(dim * 8)
        self.reduce_chan_level3 = nn.Conv2d(dim * 8, dim * 4, kernel_size=1, bias=bias)
        self.decoder_level3 = blk(dim * 4, heads[2], num_blocks[2])
        self.up3_2 = Upsample(dim * 4)
        self.reduce_chan_level2 = nn.Conv2d(dim * 4, dim * 2, kernel_size=1, bias=bias)
        self.decoder_level2 = blk(dim * 2, heads[1], num_blocks[1])
        self.up2_1 = Upsample(dim * 2)
        d1 = dim if cfg.dec1_reduce else dim * 2
        if cfg.dec1_reduce:
            self.reduce_chan_level1 = nn.Conv2d(dim * 2, dim, kernel_size=1, bias=bias)
        self.decoder_level1 = blk(d1, heads[0], num_blocks[0])
        self.refinement = blk(d1, heads[0], num_refinement_blocks)
        # ---- crude skip-connections (§3.1)
        self.skip_patch_embed1 = SkipPatchEmbed(3, 3)
        self.skip_patch_embed2 = SkipPatchEmbed(3, 3)
        self.skip_patch_embed3 = SkipPatchEmbed(3, 3)
        self.reduce_chan_level_1 = nn.Conv2d(dim * 2 + 3, dim * 2, kernel_size=1, bias=bias)
        self.reduce_chan_level_2 = nn.Conv2d(dim * 4 + 3, dim * 4, kernel_size=1, bias=bias)
        self.reduce_chan_level_3 = nn.Conv2d(dim * 8 + 3, dim * 8, kernel_size=1, bias=bias)
        self.output = nn.Conv2d(d1, out_channels, kernel_size=3, stride=1, padding=1, bias=bias)

    def forward(self, inp_img):
        e1 = self.encoder_level1(self.patch_embed(inp_img))
        s1 = self.skip_patch_embed1(inp_img)
        e2 = self.encoder_level2(self.reduce_chan_level_1(torch.cat([self.down1_2(e1), s1], 1)))
        s2 = self.skip_patch_embed2(s1)
        e3 = self.encoder_level3(self.reduce_chan_level_2(torch.cat([self.down2_3(e2), s2], 1)))
        s3 = self.skip_patch_embed3(s2)
        lat = self.latent(self.reduce_chan_level_3(torch.cat([self.down3_4(e3), s3], 1)))
        d3 = self.decoder_level3(self.reduce_chan_level3(torch.cat([self.up4_3(lat), e3], 1)))
        d2 = self.decoder_level2(self.reduce_chan_level2(torch.cat([self.up3_2(d3), e2], 1)))
        d1 = torch.cat([self.up2_1(d2), e1], 1)
        if self.cfg.dec1_reduce:
            d1 = self.reduce_chan_level1(d1)
        d1 = self.refinement(self.decoder_level1(d1))
        return self.output(d1) + inp_img

def build_model(cfg=CFG):
    return Histoformer(cfg.inp_channels, cfg.out_channels, cfg.dim, cfg.num_blocks, cfg.num_refinement_blocks,
                       cfg.heads, cfg.ffn_expansion_factor, cfg.bias, cfg.LayerNorm_type, cfg=cfg)

model = build_model(PAPER).to(DEVICE)
n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f'Histoformer (paper preset): {n_params(model)/1e6:.3f} M params')
print(f'Histoformer (repo preset) : {n_params(build_model(REPO))/1e6:.3f} M params')

Histoformer (paper preset): 16.162 M params


Histoformer (repo preset) : 16.615 M params


## 7 · Matcher / assignment
Histoformer is a dense image-to-image restoration network: every output pixel is compared with the ground-truth pixel at the same location, so there is no matcher or label-assignment step (unlike detection transformers). The closest analogue is the *internal* assignment inside DHSA — pixels are assigned to histogram bins by sorting (Section 3).

## 8 · Loss functions
*Paper §3.3, Eq. 7–10; weight α from Table 6.*

**Reconstruction (Eq. 7):** $\mathcal L_{rec}=\lVert I^{hq}-I^{gt}\rVert_1$ (mean absolute error).

**Pearson correlation (Eq. 8)** over the $3HW$ values of each image:
$$
\rho(I^{hq},I^{gt})=\frac{\sum_{i=1}^{3HW}(I^{hq}_i-\bar I^{hq})(I^{gt}_i-\bar I^{gt})}{3HW\,\sigma(I^{hq})\,\sigma(I^{gt})}
$$

**Correlation loss (Eq. 9):** $\mathcal L_{cor}=\tfrac12\big(1-\rho\big)\in[0,1]$, zero when the restored image is perfectly linearly correlated with the target.

**Total (Eq. 10):** $\mathcal L=\mathcal L_{rec}+\alpha\,\mathcal L_{cor}$ with $\alpha=1$.

In [9]:
def reconstruction_loss(pred, gt):                       # Eq. 7
    return (pred - gt).abs().mean()

def pearson_rho(pred, gt, eps=1e-12):                    # Eq. 8, one value per image
    b = pred.shape[0]
    x, y = pred.reshape(b, -1), gt.reshape(b, -1)
    xc, yc = x - x.mean(-1, keepdim=True), y - y.mean(-1, keepdim=True)
    n = x.shape[-1]
    sx = xc.pow(2).mean(-1).sqrt(); sy = yc.pow(2).mean(-1).sqrt()   # population std
    return (xc * yc).sum(-1) / (n * sx * sy + eps)

def correlation_loss(pred, gt, repo_style=False):        # Eq. 9
    if repo_style:   # official: sum-of-squares norms, +1e-6, NaN/Inf samples dropped
        b = pred.shape[0]
        x, y = pred.reshape(b, -1), gt.reshape(b, -1)
        xc, yc = x - x.mean(-1, keepdim=True), y - y.mean(-1, keepdim=True)
        rho = (xc * yc).sum(-1) / (xc.pow(2).sum(-1).sqrt() * yc.pow(2).sum(-1).sqrt() + 1e-6)
        l = (1. - rho) / 2.
        return l[~l.isnan() & ~l.isinf()].mean()
    return (0.5 * (1. - pearson_rho(pred, gt))).mean()

def total_loss(pred, gt, cfg=CFG):                       # Eq. 10
    l_rec = reconstruction_loss(pred, gt)
    l_cor = correlation_loss(pred, gt, cfg.pearson_repo_style)
    return l_rec + cfg.alpha_cor * l_cor, {'l_rec': l_rec.item(), 'l_cor': l_cor.item()}

g = torch.rand(2, 3, 32, 32)
print('rho(gt, gt)        =', pearson_rho(g, g).tolist())
print('rho(gt, 2*gt+0.1)  =', pearson_rho(2 * g + .1, g).tolist(), '(linear map -> still 1)')
print('rho(gt, 1-gt)      =', pearson_rho(1 - g, g).tolist())
print('L_cor(paper) vs L_cor(repo) on noise:', correlation_loss(torch.rand_like(g), g).item(),
      correlation_loss(torch.rand_like(g), g, True).item())

rho(gt, gt)        = [0.9999999403953552, 1.0]
rho(gt, 2*gt+0.1)  = [0.9999999403953552, 1.0] (linear map -> still 1)
rho(gt, 1-gt)      = [-1.0, -1.0000001192092896]
L_cor(paper) vs L_cor(repo) on noise: 0.5020899772644043 0.5066061019897461


## 9 · Dataset and augmentation
*Paper §4.1 "Datasets" and "Training details".*

The all-weather training set combines **9,000** Snow100K images, **1,069** RainDrop images and **9,000** Outdoor-Rain images (the TransWeather / WeatherDiff split). Testing uses Snow100K-S, Snow100K-L, Outdoor-Rain Test1 and the RainDrop test set. The repo expects paired folders `…/input/` (degraded, *lq*) and `…/gt/` (clean) with matching file names.

Training samples are random crops of `patch_size`, augmented with random horizontal and vertical flips (`aug_mode='flip'`, paper). `aug_mode='dihedral'` reproduces the repo's 8-way flip/rotate augmentation. The progressive schedule (patch 128→362 while batch 8→1) is applied by the training loop.

In [10]:
from PIL import Image

def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.float32) / 255.

def paired_random_crop(lq, gt, size):
    h, w = gt.shape[:2]
    if h < size or w < size:     # reflect-pad small images (repo `padding`)
        ph, pw = max(0, size - h), max(0, size - w)
        lq = np.pad(lq, ((0, ph), (0, pw), (0, 0)), mode='reflect'); gt = np.pad(gt, ((0, ph), (0, pw), (0, 0)), mode='reflect')
        h, w = gt.shape[:2]
    top, left = random.randint(0, h - size), random.randint(0, w - size)
    return lq[top:top + size, left:left + size], gt[top:top + size, left:left + size]

def augment_pair(lq, gt, mode='flip'):
    if mode == 'flip':                         # paper: random horizontal and vertical flips
        if random.random() < 0.5: lq, gt = lq[:, ::-1], gt[:, ::-1]
        if random.random() < 0.5: lq, gt = lq[::-1], gt[::-1]
    else:                                      # repo: one of 8 dihedral transforms
        k, flip = random.randint(0, 3), random.random() < 0.5
        if flip: lq, gt = lq[::-1], gt[::-1]
        lq, gt = np.rot90(lq, k), np.rot90(gt, k)
    return np.ascontiguousarray(lq), np.ascontiguousarray(gt)

to_tensor = lambda a: torch.from_numpy(a).permute(2, 0, 1).float()

class PairedWeatherDataset(torch.utils.data.Dataset):
    '''Paired (lq, gt) images from <root>/input and <root>/gt.'''
    def __init__(self, root, phase='train', patch_size=128, aug_mode='flip'):
        self.lq_paths = sorted(glob.glob(os.path.join(root, 'input', '*')))
        self.gt_paths = sorted(glob.glob(os.path.join(root, 'gt', '*')))
        assert len(self.lq_paths) == len(self.gt_paths), 'input/gt count mismatch'
        self.phase, self.patch_size, self.aug_mode = phase, patch_size, aug_mode
    def __len__(self): return len(self.gt_paths)
    def __getitem__(self, i):
        lq, gt = load_rgb(self.lq_paths[i]), load_rgb(self.gt_paths[i])
        if self.phase == 'train':
            lq, gt = paired_random_crop(lq, gt, self.patch_size)
            lq, gt = augment_pair(lq, gt, self.aug_mode)
        return {'lq': to_tensor(lq), 'gt': to_tensor(gt), 'name': os.path.basename(self.gt_paths[i])}

# demo on a tiny synthetic folder (stand-in for the all-weather data; nothing is trained on it)
demo_root = '/tmp/histoformer_demo'
for sub in ('input', 'gt'): os.makedirs(os.path.join(demo_root, sub), exist_ok=True)
rng = np.random.default_rng(0)
for i in range(4):
    gt_img = (rng.random((150, 170, 3)) * 255).astype(np.uint8)
    lq_img = np.clip(gt_img.astype(int) + (rng.random(gt_img.shape) < 0.05) * 200, 0, 255).astype(np.uint8)
    Image.fromarray(gt_img).save(f'{demo_root}/gt/{i}.png'); Image.fromarray(lq_img).save(f'{demo_root}/input/{i}.png')
ds = PairedWeatherDataset(demo_root, 'train', CFG.patch_size, CFG.aug_mode)
s = ds[0]
print(len(ds), 'pairs | lq', tuple(s['lq'].shape), '| gt', tuple(s['gt'].shape), '| range', float(s['lq'].min()), float(s['lq'].max()))

4 pairs | lq (3, 128, 128) | gt (3, 128, 128) | range 0.0 1.0


## 10 · Inference and evaluation metrics
*Evaluation protocol of §4.2 / Table 1, matching `Allweather/test_histoformer.py` and `compute_psnr.py`.*

* **Inference:** reflect-pad the image to a multiple of 8 (three 2× down-samplings), run the model, crop back, clamp to [0, 1].
* **Metrics:** PSNR and SSIM (11×11 Gaussian-free window, `skimage`) computed on the **Y channel** of YCbCr (MATLAB `rgb2ycbcr` coefficients), averaged over the test set.

In [11]:
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

@torch.no_grad()
def restore(model, img, factor=8):
    '''img: (B,3,H,W) in [0,1]. Reflect-pad to a multiple of `factor`, run, crop, clamp.'''
    model.eval()
    h, w = img.shape[-2:]
    ph, pw = (-h) % factor, (-w) % factor
    x = F.pad(img, (0, pw, 0, ph), mode='reflect')
    return model(x)[..., :h, :w].clamp(0, 1)

def rgb2y(img):
    '''HxWx3 float RGB in [0,1] -> Y channel in [0,1] (MATLAB rgb2ycbcr).'''
    return (img @ np.array([65.481, 128.553, 24.966]) + 16.0) / 255.

def psnr_ssim(pred, gt, y_channel=True, win=11):
    '''pred, gt: HxWx3 float arrays in [0,1].'''
    if y_channel:
        pred, gt = rgb2y(pred.astype(np.float64)), rgb2y(gt.astype(np.float64))
        ssim = structural_similarity(pred, gt, win_size=win, data_range=1)
    else:
        ssim = structural_similarity(pred, gt, win_size=win, data_range=1, channel_axis=2)
    return peak_signal_noise_ratio(gt, pred, data_range=1), ssim

@torch.no_grad()
def evaluate(model, dataset, cfg=CFG, max_images=None):
    ps, ss = [], []
    for i in range(len(dataset) if max_images is None else min(max_images, len(dataset))):
        s = dataset[i]
        out = restore(model, s['lq'][None].to(DEVICE), cfg.test_pad_factor)[0].permute(1, 2, 0).cpu().numpy()
        p, q = psnr_ssim(out, s['gt'].permute(1, 2, 0).numpy(), cfg.eval_y_channel, cfg.ssim_win)
        ps.append(p); ss.append(q)
    return float(np.mean(ps)), float(np.mean(ss))

# metric sanity: identical images -> inf PSNR / SSIM 1; known noise level -> expected PSNR
a = np.random.rand(64, 64, 3)
with np.errstate(divide='ignore'):
    print('identical  :', psnr_ssim(a, a))
b = np.clip(a + np.random.normal(0, 0.01, a.shape), 0, 1)
print('sigma=0.01 :', tuple(round(v, 3) for v in psnr_ssim(b, a, y_channel=False)), '(~40 dB expected)')
test_ds = PairedWeatherDataset(demo_root, 'test')
out = restore(build_model(CFG).to(DEVICE), test_ds[0]['lq'][None].to(DEVICE))
print('inference on a 150x170 image (not a multiple of 8) ->', tuple(out.shape))

identical  : (np.float64(inf), np.float64(1.0))
sigma=0.01 : (np.float64(40.125), np.float64(0.999)) (~40 dB expected)


inference on a 150x170 image (not a multiple of 8) -> (1, 3, 150, 170)


## 11 · Training loop (defined, not run)
*Paper §4.1 "Training details".*

* 300,000 iterations, AdamW, initial LR $3\times10^{-4}$ held for 92k iterations, then cosine-annealed to $10^{-6}$ over the remaining 208k.
* Progressive learning: starts at batch 8 / patch 128; the repo schedule continues with (5,160), (2,256), (1,320), (1,362) at 92k / 176k / 232k / 268k iterations.
* Loss: Eq. 10 with α = 1. Gradient-norm clipping at 0.01 (repo).

The cell below only defines the functions and prints the schedule; it does not train.

In [12]:
def lr_at(it, cfg=CFG):
    '''Constant 3e-4 for const_iters, then cosine to lr_min over cosine_iters.'''
    if it < cfg.const_iters: return cfg.lr
    t = min(it - cfg.const_iters, cfg.cosine_iters) / cfg.cosine_iters
    return cfg.lr_min + 0.5 * (cfg.lr - cfg.lr_min) * (1 + math.cos(math.pi * t))

def progressive_stage(it, cfg=CFG):
    bounds = np.cumsum(cfg.prog_iters)
    j = int(np.searchsorted(bounds, it, side='right'))
    j = min(j, len(cfg.prog_iters) - 1)
    return cfg.prog_batch[j], cfg.prog_patch[j]

def make_optimizer(model, cfg=CFG):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=cfg.betas, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: lr_at(it, cfg) / cfg.lr)
    return opt, sched

def train(model, train_root, cfg=CFG, val_sets=None, log_every=100, val_every=5000, ckpt_dir='checkpoints'):
    '''Full training procedure. val_sets: dict name -> PairedWeatherDataset(phase="test").'''
    os.makedirs(ckpt_dir, exist_ok=True)
    max_patch = max(cfg.prog_patch)
    ds = PairedWeatherDataset(train_root, 'train', max_patch, cfg.aug_mode)   # load at max size, crop down per stage
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=8,
                                         drop_last=True, pin_memory=True, persistent_workers=True)
    opt, sched = make_optimizer(model, cfg)
    model.train(); it = 0
    while it < cfg.total_iters:
        for batch in loader:
            bs, ps = progressive_stage(it, cfg)
            lq, gt = batch['lq'].to(DEVICE), batch['gt'].to(DEVICE)
            if bs < lq.shape[0]:                                   # sub-sample the batch
                keep = torch.randperm(lq.shape[0])[:bs]; lq, gt = lq[keep], gt[keep]
            if ps < max_patch:                                     # random sub-crop
                t, l = random.randint(0, max_patch - ps), random.randint(0, max_patch - ps)
                lq, gt = lq[..., t:t + ps, l:l + ps], gt[..., t:t + ps, l:l + ps]
            pred = model(lq)
            loss, logs = total_loss(pred, gt, cfg)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sched.step(); it += 1
            if it % log_every == 0:
                print(f'it {it:6d} | lr {sched.get_last_lr()[0]:.2e} | bs {bs} patch {ps} | '
                      f'loss {loss.item():.4f} (rec {logs["l_rec"]:.4f}, cor {logs["l_cor"]:.4f})')
            if val_sets and it % val_every == 0:
                for name, vds in val_sets.items():
                    print(f'   val {name}: PSNR/SSIM = {evaluate(model, vds, cfg)}')
                model.train()
                torch.save({'params': model.state_dict(), 'iter': it}, f'{ckpt_dir}/net_g_{it}.pth')
            if it >= cfg.total_iters: break
    return model

for it in [0, 50_000, 91_999, 92_000, 150_000, 196_000, 250_000, 299_999]:
    print(f'iter {it:7d}: lr={lr_at(it):.2e}  (batch, patch)={progressive_stage(it)}')

iter       0: lr=3.00e-04  (batch, patch)=(8, 128)
iter   50000: lr=3.00e-04  (batch, patch)=(8, 128)
iter   91999: lr=3.00e-04  (batch, patch)=(8, 128)
iter   92000: lr=3.00e-04  (batch, patch)=(5, 160)
iter  150000: lr=2.46e-04  (batch, patch)=(5, 160)
iter  196000: lr=1.50e-04  (batch, patch)=(2, 256)
iter  250000: lr=4.16e-05  (batch, patch)=(1, 320)
iter  299999: lr=1.00e-06  (batch, patch)=(1, 362)


## 12 · Sanity checks

1. Forward pass on random input for both presets; output shape and residual design.
2. Intermediate shapes at every stage (forward hooks).
3. Every loss term is finite.
4. Gradients reach the backbone (patch embedding, first and last HTB).
5. DHSA internals: sorting then scattering restores pixel order; BHR vs FHR grouping.
6. **Equivalence with the official code:** build the official `Histoformer` from the cloned repo, copy this notebook's `REPO`-preset weights into it with `strict=True`, and compare outputs on the same input.

In [13]:
torch.manual_seed(0)
x = torch.rand(2, 3, 64, 64, device=DEVICE)
for name, cfg in [('PAPER', PAPER), ('REPO', REPO)]:
    m = build_model(cfg).to(DEVICE).eval()
    with torch.no_grad(): y = m(x)
    print(f'{name:5s}: in {tuple(x.shape)} -> out {tuple(y.shape)} | finite={bool(torch.isfinite(y).all())} '
          f'| mean|out-in|={float((y - x).abs().mean()):.4f}')

PAPER: in (2, 3, 64, 64) -> out (2, 3, 64, 64) | finite=True | mean|out-in|=0.1233


REPO : in (2, 3, 64, 64) -> out (2, 3, 64, 64) | finite=True | mean|out-in|=0.1047


In [14]:
# 2. shapes at every stage
m = build_model(PAPER).to(DEVICE).eval()
shapes = {}
names = ['patch_embed', 'encoder_level1', 'encoder_level2', 'encoder_level3', 'latent',
         'decoder_level3', 'decoder_level2', 'decoder_level1', 'refinement', 'output']
hooks = [getattr(m, n).register_forward_hook(lambda mod, i, o, n=n: shapes.__setitem__(n, tuple(o.shape))) for n in names]
with torch.no_grad(): m(torch.rand(1, 3, 128, 128, device=DEVICE))
for h in hooks: h.remove()
for n in names: print(f'{n:16s} {shapes[n]}')

patch_embed      (1, 36, 128, 128)
encoder_level1   (1, 36, 128, 128)
encoder_level2   (1, 72, 64, 64)
encoder_level3   (1, 144, 32, 32)
latent           (1, 288, 16, 16)
decoder_level3   (1, 144, 32, 32)
decoder_level2   (1, 72, 64, 64)
decoder_level1   (1, 36, 128, 128)
refinement       (1, 36, 128, 128)
output           (1, 3, 128, 128)


In [15]:
# 3 + 4. losses finite and gradients reach the backbone
for name, cfg in [('PAPER', PAPER), ('REPO', REPO)]:
    torch.manual_seed(1)
    m = build_model(cfg).to(DEVICE).train()
    lq, gt = torch.rand(2, 3, 64, 64, device=DEVICE), torch.rand(2, 3, 64, 64, device=DEVICE)
    pred = m(lq)
    loss, logs = total_loss(pred, gt, cfg)
    loss.backward()
    terms = {'L_rec': logs['l_rec'], 'L_cor': logs['l_cor'], 'L_total': loss.item()}
    print(f'[{name}] ' + ', '.join(f'{k}={v:.4f}' for k, v in terms.items()),
          '| all finite:', all(math.isfinite(v) for v in terms.values()))
    probes = {'patch_embed.proj': m.patch_embed.proj.weight,
              'encoder_level1[0].attn_g.qkv': m.encoder_level1[0].attn_g.qkv.weight,
              'latent[-1].ffn.project_in': m.latent[-1].ffn.project_in.weight,
              'skip_patch_embed3 conv': m.skip_patch_embed3.proj[1].weight,
              'refinement[-1].attn_g.project_out': m.refinement[-1].attn_g.project_out.weight}
    for k, p in probes.items():
        print(f'    grad |{k:36s}| = {p.grad.norm().item():.3e}')
    no_grad = [n for n, p in m.named_parameters() if p.grad is None]
    print('    params without grad:', len(no_grad), '(temperature is only used by attn_mode="repo")' if no_grad else '')

[PAPER] L_rec=0.3457, L_cor=0.5044, L_total=0.8501 | all finite: True
    grad |patch_embed.proj                    | = 2.973e+01
    grad |encoder_level1[0].attn_g.qkv        | = 3.417e+01
    grad |latent[-1].ffn.project_in           | = 2.462e-01
    grad |skip_patch_embed3 conv              | = 2.667e-03
    grad |refinement[-1].attn_g.project_out   | = 1.214e-02
    params without grad: 40 (temperature is only used by attn_mode="repo")


[REPO] L_rec=0.3453, L_cor=0.4996, L_total=0.8449 | all finite: True
    grad |patch_embed.proj                    | = 1.041e-01
    grad |encoder_level1[0].attn_g.qkv        | = 1.506e-03
    grad |latent[-1].ffn.project_in           | = 2.397e-03
    grad |skip_patch_embed3 conv              | = 5.000e-05
    grad |refinement[-1].attn_g.project_out   | = 1.608e-02
    params without grad: 0 


In [16]:
# 5. DHSA internals
torch.manual_seed(0)
att = Attention_histogram(4, num_heads=2, bias=False)
v = torch.randn(1, 4, 4, 4)
vs, idx = v.reshape(1, 4, -1).sort(-1)
print('scatter(sorted, idx) == original:', torch.equal(torch.scatter(vs, 2, idx, vs).view_as(v), v))
seq = torch.arange(8.).view(1, 1, 8)
print('sorted sequence     :', seq[0, 0].tolist())
print('BHR tokens = B=2 bins of HW/B consecutive values:', rearrange(seq, 'b c (factor hw) -> b (c factor) hw', factor=2)[0].tolist())
print('FHR tokens = slot j of every bin of B=2 neighbours {0,1},{2,3},..:', rearrange(seq, 'b c (hw factor) -> b (c factor) hw', factor=2)[0].tolist())

scatter(sorted, idx) == original: True
sorted sequence     : [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
BHR tokens = B=2 bins of HW/B consecutive values: [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]]
FHR tokens = slot j of every bin of B=2 neighbours {0,1},{2,3},..: [[0.0, 2.0, 4.0, 6.0], [1.0, 3.0, 5.0, 7.0]]


In [17]:
# 6. equivalence with the official implementation (same weights, same input)
import importlib.util, sys
REPO_ARCH = '/home/claude/repo_dl/Histoformer/basicsr/models/archs/histoformer_arch.py'   # path to the cloned repo
if os.path.exists(REPO_ARCH):
    spec = importlib.util.spec_from_file_location('official_histoformer_arch', REPO_ARCH)
    official = importlib.util.module_from_spec(spec); spec.loader.exec_module(official)
    torch.manual_seed(123)
    ours = build_model(REPO).to(DEVICE).eval()
    ref = official.Histoformer(inp_channels=3, out_channels=3, dim=36, num_blocks=[4, 4, 6, 8],
                               num_refinement_blocks=4, heads=[1, 2, 4, 8], ffn_expansion_factor=2.667,
                               bias=False, LayerNorm_type='WithBias').to(DEVICE).eval()
    # randomise temperatures so the 'repo' attention path is actually exercised
    with torch.no_grad():
        for n, p in ours.named_parameters():
            if n.endswith('temperature'): p.uniform_(0.5, 2.0)
    missing = ref.load_state_dict(ours.state_dict(), strict=True)
    print('state_dict keys identical; strict load:', missing)
    for hw in [(64, 64), (72, 96), (128, 128)]:
        x = torch.rand(1, 3, *hw, device=DEVICE)
        with torch.no_grad(): a, b = ours(x), ref(x.clone())
        print(f'input {hw}: max |ours - official| = {(a - b).abs().max().item():.3e}')
    # paper-preset model differs from the repo by design (switches 1-3); show it is a different function
    paper_m = build_model(replace(PAPER, dec1_reduce=False)).to(DEVICE).eval()
    paper_m.load_state_dict(ours.state_dict(), strict=True)
    with torch.no_grad(): d = (paper_m(x) - ours(x)).abs().max().item()
    print(f'paper-mode attention/unsort with the same weights: max diff = {d:.3e} (expected > 0)')
else:
    print('Official repo not found at', REPO_ARCH, '- clone https://github.com/sunshangquan/Histoformer to run this check.')

state_dict keys identical; strict load: <All keys matched successfully>


input (64, 64): max |ours - official| = 0.000e+00


input (72, 96): max |ours - official| = 0.000e+00


input (128, 128): max |ours - official| = 0.000e+00


paper-mode attention/unsort with the same weights: max diff = 6.782e-01 (expected > 0)


### Loading the official checkpoint
The released all-weather weights (`net_g_best.pth`, linked from the repo README) load into the `REPO` preset:

```python
model = build_model(REPO).to(DEVICE)
ckpt = torch.load('Allweather/pretrained_models/net_g_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['params'], strict=True)
psnr, ssim = evaluate(model, PairedWeatherDataset('<data>/test/Snow100K-L', 'test'), REPO)
```
The parameter names are identical (verified above with `strict=True`), so no key remapping is needed.